# 3. Snowflake: Import the Databricks Ossie from S3

This notebook reads the Ossie file that Databricks produced from S3 and creates a
semantic view from it. The measure added in Databricks (`TOTAL_QUANTITY`) comes across.

The import is a single built-in function -- Snowflake reads Ossie natively.
No file download or upload needed; both platforms share the same S3 bucket.

## Step 1 - Set your database and schema

In [ ]:
DATABASE = "DEMOS"
SCHEMA   = "EXT_SEMANTIC_INTEROP"
print(f"Working in {DATABASE}.{SCHEMA}")

In [ ]:
USE ROLE ACCOUNTADMIN;
USE SCHEMA {{DATABASE}}.{{SCHEMA}};

## Step 2 - Read the Databricks Ossie from S3

The file is at `s3://snowflake-ossie-interop/ossie/ossie_from_databricks.yaml`,
accessible via the external stage.

In [ ]:
LIST @{{DATABASE}}.{{SCHEMA}}.OSSIE_S3_STAGE;

In [ ]:
SET yaml_content = (
  SELECT $1 FROM @{{DATABASE}}.{{SCHEMA}}.OSSIE_S3_STAGE/ossie_from_databricks.yaml
  (FILE_FORMAT => '{{DATABASE}}.{{SCHEMA}}.RAW_TEXT_FMT')
);

In [ ]:
SELECT $yaml_content;

## Step 3 - Choose the target view name

Default `SALES_SV_V2` leaves the original `SALES_SV` untouched.

In [ ]:
SET target_view = 'SALES_SV_V2';

In [ ]:
SET yaml_to_import = (SELECT REPLACE($yaml_content, 'SALES_SV_V2', $target_view));

## Step 4 - Import

In [ ]:
CALL SYSTEM$CREATE_SEMANTIC_VIEW_FROM_OSSIE_YAML('{{DATABASE}}.{{SCHEMA}}', $yaml_to_import);

## Step 5 - Verify

The round-tripped view returns the same numbers as Databricks: EAST 12/750/5, WEST 11/700/5.

In [ ]:
SET target_fqn = '{{DATABASE}}.{{SCHEMA}}.' || $target_view;
SELECT * FROM SEMANTIC_VIEW(
  IDENTIFIER($target_fqn)
  DIMENSIONS region
  METRICS total_quantity, total_order_amount, order_count
) ORDER BY region;